In [ ]:
import pandas as pd
import numpy as np

ruta = "../data/tcga_simple_train.csv"
df = pd.read_csv(ruta)

df.head

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

distribucion = df['t'].value_counts(normalize=True).sort_index() * 100

print("Probabilidad de cada clase (en %):")
print(distribucion)

plt.figure(figsize=(10, 6))
orden = ['T1', 'T2', 'T3', 'T4']

ax = sns.countplot(data=df, x='t', order=orden, palette='magma')

for p in ax.patches:
    percentage = f'{100 * p.get_height() / len(df):.1f}%'
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.title('Distribución del Estadio del Tumor (Variable Objetivo)', fontsize=15)
plt.xlabel('Estadio T', fontsize=12)
plt.ylabel('Número de Informes', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='t', y='word_count', order=orden, palette='viridis')
plt.title('Longitud de Informes por Estadio T')
plt.xlabel('Estadio')
plt.ylabel('Número de Palabras')
plt.show()

In [ ]:
import re
def count_measures(text):
    return len(re.findall(r'\d+(?:\.\d+)?\s*(?:cm|mm)', str(text).lower()))

df['measure_count'] = df['text'].apply(count_measures)
print(df.groupby('t')['measure_count'].mean())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngram(corpus, n=None):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:10]


top_bigrams_t1 = get_top_ngram(df[df['t']=='T1']['text'], n=2)
top_bigrams_t2 = get_top_ngram(df[df['t']=='T2']['text'], n=2)
top_bigrams_t3 = get_top_ngram(df[df['t']=='T3']['text'], n=2)
top_bigrams_t4 = get_top_ngram(df[df['t']=='T4']['text'], n=2)

print("Top 10 bigramas para T1:", top_bigrams_t1)
print("Top 10 bigramas para T2:", top_bigrams_t2)
print("Top 10 bigramas para T3:", top_bigrams_t3)
print("Top 10 bigramas para T4:", top_bigrams_t4)